In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

## Obteniendo los parámetros

In [0]:
%sql
CREATE WIDGET TEXT catalog_name DEFAULT 'smartdata_project';
CREATE WIDGET TEXT schema_source DEFAULT 'bronze';
CREATE WIDGET TEXT schema_sink DEFAULT 'silver';
CREATE WIDGET TEXT source_persons DEFAULT 'bronze_persons';
CREATE WIDGET TEXT sink_persons DEFAULT 'silver_persons';
CREATE WIDGET TEXT w_persons DEFAULT '1900-01-01';

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
schema_source = dbutils.widgets.get('schema_source')
schema_sink = dbutils.widgets.get('schema_sink')
source_persons = dbutils.widgets.get('source_persons')
sink_persons = dbutils.widgets.get('sink_persons')
w_persons = dbutils.widgets.get('w_persons')

In [0]:
source_persons_table = f'{catalog_name}.{schema_source}.{source_persons}'
df_persons = spark.sql(f"SELECT * FROM {source_persons_table} WHERE updated_at > '{w_persons}'")

### Validando longitud de city y age

In [0]:
df_persons_dep = (
    df_persons
    .withColumn('rn', F.row_number().over(Window.partitionBy('customer_id_original').orderBy(F.col('updated_at').desc())))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

In [0]:
df_persons_dep = (
    df_persons_dep
    .withColumn('age', F.col('age').cast('int'))
    .withColumn('zipcode', F.col('zipcode').cast('string'))
    .withColumn('full_name', F.initcap(F.trim(F.col('full_name'))))
    .withColumn('city', F.initcap(F.trim(F.col('city'))))
).withColumnRenamed('customer_id_original','customer_id')

In [0]:
df_persons_invalid = df_persons_dep.filter(
    (F.col('age') > 100) |
    (F.col('age') < 18) |
    F.col('full_name').isNull() |
    (F.length(F.col('state')) < 2) |
    (F.col('rfm_id') > 6) |
    (F.col('rfm_id') < 1)
)

invalid_condition = (
    (F.col('age') > 100) |
    (F.col('age') < 18) |
    F.col('full_name').isNull() |
    (F.length(F.col('state')) < 2) |
    (F.col('rfm_id') > 6) |
    (F.col('rfm_id') < 1)
)

invalid_count = df_persons_invalid.count()
if invalid_count > 0:
    print(f"⚠️ {invalid_count} registros inválidos:")

    df_persons_quarantine = (
        df_persons_dep
        .filter(invalid_condition)
        .withColumn('rejection_reason',
            F.when((F.col('age') > 100) | (F.col('age') < 18), 'age out of range (18-100)')
            .when(F.col('full_name').isNull(), 'full_name is null')
            .when(F.length(F.col('state')) < 2, 'state too short')
            .when((F.col('rfm_id') > 6) | (F.col('rfm_id') < 1), 'rfm_id out of range (1-6)')
            .otherwise(F.lit('unknown'))
        )
        .withColumn('rfm_id', F.col('rfm_id').cast('string'))
        .withColumn('quarantined_at', F.current_timestamp())
    )

    df_persons_quarantine.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog_name}.{schema_sink}.quarantine_persons")

    df_silver_persons = df_persons_dep.filter(~invalid_condition)

else:
    print("✅ Todos los registros son válidos")

In [0]:
target_person_table = f'{catalog_name}.{schema_sink}.{sink_persons}'
if spark.catalog.tableExists(target_person_table):

    source_table = DeltaTable.forName(spark, target_person_table)

    (
        source_table.alias("target")
        .merge(
            df_silver_persons.alias("source"),
            f"target.customer_id = source.customer_id"
        )
        # Registro existe y cambió algo → actualizar
        .whenMatchedUpdateAll()
        # Registro nuevo → insertar
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        df_silver_persons.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .clusterBy('customer_id','updated_at','city')  # Liquid Clustering
        .saveAsTable(target_person_table)
    )